# S0002 Splits

In [ ]:
import pandas as pd
import tqdm
import os
import matplotlib.pyplot as plt
import seaborn as sns
site = 'S0002' # S0001 or S0002
matched_eeg_report_recording_save_path = f'[PATH_TO_MATCHED_EEG_RECORDINGS_REPORT]/{site}'


# read description_df
description_df = pd.read_csv(os.path.join(matched_eeg_report_recording_save_path, f'{site}_matched_eeg_report_recording_description.csv'))
print(description_df['NumberOfSessions'].value_counts())
print(description_df['VisitTypeDSC'].value_counts())
print(description_df['Number_of_Extracted_EEG_sections'].value_counts())
print(description_df['Number_of_Extracted_Clinical_sections'].value_counts())
print(description_df['Number_of_Empty_EEG_sections'].value_counts())
print(description_df['Number_of_Empty_Clinical_sections'].value_counts())
description_df
#

In [ ]:
# get all unique EEG sections
unique_eeg_sections = []

for index, row in tqdm.tqdm(description_df.iterrows(), total=len(description_df)):
    eeg_sections = row['Extracted_EEG_sections']
    eeg_sections = str(eeg_sections).split(',')
    for eeg_section in eeg_sections:
        eeg_section = eeg_section.lower().strip()
        if eeg_section not in unique_eeg_sections:
            unique_eeg_sections.append(eeg_section)

print(len(unique_eeg_sections))
print(unique_eeg_sections)

In [ ]:
import ast

mismatched = description_df[
    description_df['NumberOfSessions'] != description_df['Processed_EEG_Paths'].apply(
        lambda x: len(str(x).split(','))
    )
]
mismatched
# mismatched

In [ ]:
SECTION_STANDARDIZATION_MAPPING = {
    'details:': 'EEG DESCRIPTION/DETAILS',
    'detail:': 'EEG DESCRIPTION/DETAILS',
    'description:': 'EEG DESCRIPTION/DETAILS',
    
    'impression:': 'IMPRESSION/INTERPRETATION',
    'interpretation:': 'IMPRESSION/INTERPRETATION',
    
    'background:': 'BACKGROUND ACTIVITY',
    'background activity:': 'BACKGROUND ACTIVITY',
    
    
    'seizures:' : 'SEIZURES',
    'events/seizures:': 'EVENTS/SEIZURES',
    
    'epileptiform abnormalities:': 'EPLEPTIFORM ABNORMALITIES',
    'interictal epileptiform abnormalities:': 'INTERICTAL EPLEPTIFORM ABNORMALITIES',
    'sleep:': 'SLEEP',
}

# map the Extracted_EEG_sections in description_df to the standardized sections 

description_df['Standardized_EEG_sections'] = description_df['Extracted_EEG_sections'].apply(lambda x: ','.join([SECTION_STANDARDIZATION_MAPPING[section.lower().strip()] for section in str(x).split(',')]))
description_df


In [ ]:
# check if there are any rows with repeated sections in Standardized_EEG_sections
repeated_rows = description_df[description_df['Standardized_EEG_sections'].apply(lambda x: len(str(x).split(',')) != len(set(str(x).split(','))))]
repeated_rows[['Extracted_EEG_sections','Standardized_EEG_sections']]

for index, row in repeated_rows.iterrows():
    print(row['VisitTypeDSC'])
    print(row['Extracted_EEG_sections'])
    print(row['Standardized_EEG_sections'])
    print('-'*100)

In [ ]:
# remove repeated rows in description_df
description_df = description_df[description_df['Standardized_EEG_sections'].apply(lambda x: len(str(x).split(',')) == len(set(str(x).split(','))))]
print(description_df.shape)

In [ ]:
# check if there are any rows with repeated sections in Standardized_EEG_sections
repeated_rows = description_df[description_df['Standardized_EEG_sections'].apply(lambda x: len(str(x).split(',')) != len(set(str(x).split(','))))]
repeated_rows[['Extracted_EEG_sections','Standardized_EEG_sections']]

for index, row in repeated_rows.iterrows():
    print(row['VisitTypeDSC'])
    print(row['Extracted_EEG_sections'])
    print(row['Standardized_EEG_sections'])
    print('-'*100)

# Randomly Split the data of VisitTypeDsc == 'EEG' to train val and test

In [ ]:
visit_type_list = ['EEG']
number_of_sessions_limit = 1

VisitType_EEG_df = description_df[description_df['VisitTypeDSC'].isin(visit_type_list)]
print(VisitType_EEG_df.shape)

# remove the rows where NumberOfSessions >1
VisitType_EEG_df = VisitType_EEG_df[VisitType_EEG_df['NumberOfSessions']<=number_of_sessions_limit]

print(VisitType_EEG_df.shape)

# get all unique BDSPPatientID
unique_patients = VisitType_EEG_df['BDSPPatientID'].unique()
print(len(unique_patients))

# deterministic shuffle the unique_patients and split into 60% 20% 20%
np.random.seed(42)
np.random.shuffle(unique_patients)
train_patients = unique_patients[:int(len(unique_patients)*0.6)]
val_patients = unique_patients[int(len(unique_patients)*0.6):int(len(unique_patients)*0.8)]
test_patients = unique_patients[int(len(unique_patients)*0.8):]

# print the length of each split
print(len(train_patients), len(val_patients), len(test_patients))

train_S0002_df = description_df[description_df['BDSPPatientID'].isin(train_patients)]
train_S0002_df = train_S0002_df[train_S0002_df['VisitTypeDSC'].isin(visit_type_list)]
train_S0002_df = train_S0002_df[train_S0002_df['NumberOfSessions']<=number_of_sessions_limit]

val_S0002_df = description_df[description_df['BDSPPatientID'].isin(val_patients)]
val_S0002_df = val_S0002_df[val_S0002_df['VisitTypeDSC'].isin(visit_type_list)]
val_S0002_df = val_S0002_df[val_S0002_df['NumberOfSessions']<=number_of_sessions_limit]
    
test_S0002_df = description_df[description_df['BDSPPatientID'].isin(test_patients)]
test_S0002_df = test_S0002_df[test_S0002_df['VisitTypeDSC'].isin(visit_type_list)]
test_S0002_df = test_S0002_df[test_S0002_df['NumberOfSessions']<=number_of_sessions_limit]

print(train_S0002_df.shape, val_S0002_df.shape, test_S0002_df.shape)
print(train_S0002_df['VisitTypeDSC'].value_counts(), val_S0002_df['VisitTypeDSC'].value_counts(), test_S0002_df['VisitTypeDSC'].value_counts())
# drop VisitTypeDSC ProcedureDSC RecordType AgeAtVisit SexDSC ProcedureDSC(Reports) Empty_EEG_sections Number_of_Empty_EEG_sections Empty_Clinical_sections Number_of_Empty_Clinical_sections 
train_S0002_df = train_S0002_df.drop(columns=['ProcedureDSC', 'RecordType', 'AgeAtVisit', 'SexDSC', 'ProcedureDSC(Reports)', 'Empty_EEG_sections', 'Number_of_Empty_EEG_sections', 'Empty_Clinical_sections', 'Number_of_Empty_Clinical_sections'])
val_S0002_df = val_S0002_df.drop(columns=[ 'ProcedureDSC', 'RecordType', 'AgeAtVisit', 'SexDSC', 'ProcedureDSC(Reports)', 'Empty_EEG_sections', 'Number_of_Empty_EEG_sections', 'Empty_Clinical_sections', 'Number_of_Empty_Clinical_sections'])
test_S0002_df = test_S0002_df.drop(columns=['ProcedureDSC', 'RecordType', 'AgeAtVisit', 'SexDSC', 'ProcedureDSC(Reports)', 'Empty_EEG_sections', 'Number_of_Empty_EEG_sections', 'Empty_Clinical_sections', 'Number_of_Empty_Clinical_sections'])

print(train_S0002_df.shape, val_S0002_df.shape, test_S0002_df.shape)

# reset the index
train_S0002_df = train_S0002_df.reset_index(drop=True)
val_S0002_df = val_S0002_df.reset_index(drop=True)
test_S0002_df = test_S0002_df.reset_index(drop=True)

# save the data
save_path = os.path.join('[PATH_TO_PROCESSED_REPORTS]', 'random_split_data_by_patient')
os.makedirs(save_path, exist_ok=True)
# train_S0002_df.to_csv(os.path.join(save_path,  f'{site}_train_split.csv'), index=False)
# val_S0002_df.to_csv(os.path.join(save_path,  f'{site}_val_split.csv'), index=False)
# test_S0002_df.to_csv(os.path.join(save_path,  f'{site}_test_split.csv'), index=False)
